# Bromobutane RESP Example
## Without Extra Points, and
## With an Extra Point that Models a Sigma Hole

---

Specs:
- **2 conformation**
- **8 spatial orientation** (i.e., 2 conf. x 4 spatial rotations)
- Two-stage RESP is performed
    - Stage 1: stage_1.ini
    - Stage 2: stage_2.ini

---

In [1]:
import os
from pathlib import Path
import sys

import numpy as np
from scipy.spatial.transform import Rotation as R

from resp_ep import driver
from resp_ep import extras

In [2]:
def charges2freeze_format(atoms: list, charges_array: np.array):
    """ Formats and prints partial charges for specific atoms as constraints.

        This function extracts atomic charges from a multidimensional array 
        and formats them into a specific string structure required for freezing 
        charges during geometry or charge optimizations.

        Args:
            atoms (list): A list of atom numbers (1-indexed) to format.
    
        Globals Used:
            freeze_charges (iterable): 1-indexed atom numbers to look up.
            charges_stage_1_x (list/tuple of np.ndarray): Structure holding the
                charge arrays, where index [1] contains the target stage charges.

        Returns:
            None: The function prints the formatted string directly to stdout.

        Example:
            >>> charges2freeze_format([5, 6, 7])
            constraint_charge :  5 =  0.09735144,
                                 6 = -0.01063565,
                                 7 = -0.00799172
    """
    lines = []
    for atom in atoms:
        index = atom - 1
        charge = charges_array[1][index]
        lines.append(f"{atom:>3} = {charge: .8f}")

    output = "constraint_charge :" + ",\n                   ".join(lines)

    print(output)

## Create rotational variants of the input

- Probably not needed since you already have teh *.dat files from a previous RESP calculstions.

In [3]:
def rotate_xyz(xyz: list, x_rotation: float, y_rotation: float, z_rotation: float) -> list:
    """ Rotates 3D Cartesian coordinates sequentially around the X, Y, and Z axes.

        Args:
            xyz: A list of [x, y, z] coordinates to rotate.
            x_rotation: Rotation angle around the X-axis in degrees.
            y_rotation: Rotation angle around the Y-axis in degrees.
            z_rotation: Rotation angle around the Z-axis in degrees.

        Returns:
            A list of the newly rotated [x, y, z] coordinates.
    """
    rotation = R.from_euler('xyz', [x_rotation, y_rotation, z_rotation], degrees=True)
    return rotation.apply(xyz).tolist()


def generate_rotamers(basename, num_variants=4, step=72):
    """ Generates unique spatial variants of a molecule by cycling isolated axis rotations.

    Args:
        basename: The prefix name of the input/output .xyz files.
        num_variants: Total number of rotated conformers to generate.
        step: The increment angle in degrees used to step through rotations.

    Warning:
        High values for num_variants in combo with the step size may result in 
        duplicate spatial orientations. Since the rotations are cyclic, any total axis 
        rotation that reaches or exceeds 360 degrees will replicate a previously 
        generated conformer.
    """
    input_file = f'{basename}.xyz'
    atoms, xyz_coords = extras.parse_xyz(input_file)

    alphabet = "abcdefghijklmnopqrstuvwxyz"
    suffixes = alphabet[0 : num_variants]  # e.g., "a, b, c, d" if num_variants=4

    for i, suffix in enumerate(suffixes):
        output_file = f'{basename}_{suffix}.xyz'

        # Maximize differences: Isolate rotations to distinct axes per file
        # conf. a: [0, 0, 0], conf. b: [72, 0, 0], conf c: [0, 144, 0], conf d: [216, 0, 0, ]
        angle = step * i
        rotations = [0, 0, 0]
        rotations[i % 3] = angle

        new_coords = [rotate_xyz(xyz, *rotations) for xyz in xyz_coords]

        extras.write_xyz(elements=atoms, xyz=new_coords, output_filepath=output_file)
        print(f"Generated: {output_file} with rotations {rotations}")

In [4]:
conformers = ['1_bromobutane_conf1', '1_bromobutane_conf1_x', '1_bromobutane_conf2', '1_bromobutane_conf2_x']

for conformer in conformers:
    generate_rotamers(basename=conformer, num_variants=4)

Generated: 1_bromobutane_conf1_a.xyz with rotations [0, 0, 0]
Generated: 1_bromobutane_conf1_b.xyz with rotations [0, 72, 0]
Generated: 1_bromobutane_conf1_c.xyz with rotations [0, 0, 144]
Generated: 1_bromobutane_conf1_d.xyz with rotations [216, 0, 0]
Generated: 1_bromobutane_conf1_x_a.xyz with rotations [0, 0, 0]
Generated: 1_bromobutane_conf1_x_b.xyz with rotations [0, 72, 0]
Generated: 1_bromobutane_conf1_x_c.xyz with rotations [0, 0, 144]
Generated: 1_bromobutane_conf1_x_d.xyz with rotations [216, 0, 0]
Generated: 1_bromobutane_conf2_a.xyz with rotations [0, 0, 0]
Generated: 1_bromobutane_conf2_b.xyz with rotations [0, 72, 0]
Generated: 1_bromobutane_conf2_c.xyz with rotations [0, 0, 144]
Generated: 1_bromobutane_conf2_d.xyz with rotations [216, 0, 0]
Generated: 1_bromobutane_conf2_x_a.xyz with rotations [0, 0, 0]
Generated: 1_bromobutane_conf2_x_b.xyz with rotations [0, 72, 0]
Generated: 1_bromobutane_conf2_x_c.xyz with rotations [0, 0, 144]
Generated: 1_bromobutane_conf2_x_d.xyz

## Stage 1

In [5]:
stage_1 = f'''\
[molecules]
input_files: 1_bromobutane_conf1_a.xyz,
             1_bromobutane_conf1_b.xyz,
             1_bromobutane_conf1_c.xyz,
             1_bromobutane_conf1_d.xyz,
             1_bromobutane_conf2_a.xyz,
             1_bromobutane_conf2_b.xyz,
             1_bromobutane_conf2_c.xyz,
             1_bromobutane_conf2_d.xyz
    
[vdw.surface.options]
vdw_scale_factors : 1.4, 1.6, 1.8, 2.0
vdw_point_density : 1.0
esp               : None
grid              : None
vdw_radii_set     : legacy
vdw_radii         : BR = 1.85

[hyperbolic.restraint.options]
weight            : 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0
restraint         : True
resp_a            : 0.0005
resp_b            : 0.1
ihfree            : True
toler             : 1e-5
max_it            : 50 

[constraints]
constraint_charge : None
equivalent_groups : None

[qm.options]
method_esp : hf
basis_esp  : assign H 6-31G*,
             assign C 6-31G*,
             assign O 6-31G*,
             assign Br 6-31G*

formal_charge : 0
multiplicity  : 1

'''

with open('stage_1.ini', 'w') as file:
    file.write(stage_1)

#### Extra Points

In [6]:
stage_1_x = f'''\
[molecules]
input_files: 1_bromobutane_conf1_x_a.xyz,
             1_bromobutane_conf1_x_b.xyz,
             1_bromobutane_conf1_x_c.xyz,
             1_bromobutane_conf1_x_d.xyz,
             1_bromobutane_conf2_x_a.xyz,
             1_bromobutane_conf2_x_b.xyz,
             1_bromobutane_conf2_x_c.xyz,
             1_bromobutane_conf2_x_d.xyz
    
[vdw.surface.options]
vdw_scale_factors : 1.4, 1.6, 1.8, 2.0
vdw_point_density : 1.0
esp               : 1_bromobutane_conf1_a_esp.dat,
                    1_bromobutane_conf1_b_esp.dat,
                    1_bromobutane_conf1_c_esp.dat,
                    1_bromobutane_conf1_d_esp.dat,
                    1_bromobutane_conf2_a_esp.dat,
                    1_bromobutane_conf2_b_esp.dat,
                    1_bromobutane_conf2_c_esp.dat,
                    1_bromobutane_conf2_d_esp.dat

grid              : 1_bromobutane_conf1_a_grid.dat,
                    1_bromobutane_conf1_b_grid.dat,
                    1_bromobutane_conf1_c_grid.dat,
                    1_bromobutane_conf1_d_grid.dat,
                    1_bromobutane_conf2_a_grid.dat,
                    1_bromobutane_conf2_b_grid.dat,
                    1_bromobutane_conf2_c_grid.dat,
                    1_bromobutane_conf2_d_grid.dat

vdw_radii_set     : legacy
vdw_radii         : BR = 1.85,
                    X = 0.0

[hyperbolic.restraint.options]
weight            : 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0
restraint         : True
resp_a            : 0.0005
resp_b            : 0.1
ihfree            : True
toler             : 1e-5
max_it            : 50 

[constraints]
constraint_charge : None
equivalent_groups : None

[qm.options]
method_esp : None
basis_esp  : None

formal_charge : 0
multiplicity  : 1
'''

with open('stage_1_x.ini', 'w') as file:
    file.write(stage_1_x)

In [7]:
charges_stage_1 = driver.resp(input_ini='stage_1.ini')
print()
print(f'ESP:\n{charges_stage_1[0]}\n')
print(f'RESP Stage 1:\n{charges_stage_1[1]}')


Determining Partial Atomic Charges

INPUT COORDS, ANGSTROMS
 [[-3.91966085  0.13035288 -0.25795093]
 [-4.74768397  0.48149055 -0.86502369]
 [-4.01597049 -0.94643743 -0.15602488]
 [-4.02396557  0.56750725  0.73056208]
 [-2.57911804  0.50662453 -0.88838478]
 [-2.52675434  1.58639862 -1.00772124]
 [-2.51881864  0.08285356 -1.88822254]
 [-1.38654736  0.02487902 -0.05501505]
 [-1.42860031 -1.05333239  0.06020295]
 [-1.43653557  0.45157596  0.94148564]
 [-0.0679235   0.41186001 -0.70396561]
 [ 0.0550455  -0.03724304 -1.67646333]
 [ 0.04702298  1.48067197 -0.78759998]
 [ 1.47236279 -0.19664039  0.34908209]]
PSI4:  "['assign H 6-31G*', 'assign C 6-31G*', 'assign O 6-31G*', 'assign Br 6-31G*']"
Points: 827; Coord: 14
INPUT COORDS, ANGSTROMS
 [[-1.45656773  0.13035288  3.64810777]
 [-2.28980145  0.48149055  4.24800876]
 [-1.3893916  -0.94643743  3.77120056]
 [-0.54866792  0.56750725  4.05277477]
 [-1.64189543  0.50662453  2.17836102]
 [-1.73920989  1.58639862  2.09168319]
 [-2.57416412  0.08285

In [8]:
charges_stage_1_x = driver.resp(input_ini='stage_1_x.ini')
print()
print(f'ESP:\n{charges_stage_1_x[0]}\n')
print(f'RESP Stage 1:\n{charges_stage_1_x[1]}')


Determining Partial Atomic Charges

INPUT COORDS, ANGSTROMS
 [[-3.91966085  0.13035288 -0.25795093]
 [-4.74768397  0.48149055 -0.86502369]
 [-4.01597049 -0.94643743 -0.15602488]
 [-4.02396557  0.56750725  0.73056208]
 [-2.57911804  0.50662453 -0.88838478]
 [-2.52675434  1.58639862 -1.00772124]
 [-2.51881864  0.08285356 -1.88822254]
 [-1.38654736  0.02487902 -0.05501505]
 [-1.42860031 -1.05333239  0.06020295]
 [-1.43653557  0.45157596  0.94148564]
 [-0.0679235   0.41186001 -0.70396561]
 [ 0.0550455  -0.03724304 -1.67646333]
 [ 0.04702298  1.48067197 -0.78759998]
 [ 1.47236279 -0.19664039  0.34908209]
 [ 2.49264568 -0.59970997  1.04661901]]
Points: 827; Coord: 15
INPUT COORDS, ANGSTROMS
 [[-1.45656773  0.13035288  3.64810777]
 [-2.28980145  0.48149055  4.24800876]
 [-1.3893916  -0.94643743  3.77120056]
 [-0.54866792  0.56750725  4.05277477]
 [-1.64189543  0.50662453  2.17836102]
 [-1.73920989  1.58639862  2.09168319]
 [-2.57416412  0.08285356  1.81204603]
 [-0.48078912  0.02487902  1.30

### Extract charges from stage 1 that you want to freeze

In [9]:
charges2freeze_format(atoms = [14], charges_array=charges_stage_1) #Atom/Extra Point #s

constraint_charge : 14 = -0.20276883


In [10]:
charges2freeze_format(atoms = [14, 15], charges_array=charges_stage_1_x) #Atom/Extra Point #s

constraint_charge : 14 = -0.41354762,
                    15 =  0.10314887


Copy and paste the above lines in the INI files below.

## Stage 2

In [11]:
stage_2 = f'''\
[molecules]
input_files: 1_bromobutane_conf1_a.xyz,
             1_bromobutane_conf1_b.xyz,
             1_bromobutane_conf1_c.xyz,
             1_bromobutane_conf1_d.xyz,
             1_bromobutane_conf2_a.xyz,
             1_bromobutane_conf2_b.xyz,
             1_bromobutane_conf2_c.xyz,
             1_bromobutane_conf2_d.xyz
    
[vdw.surface.options]
vdw_scale_factors : 1.4, 1.6, 1.8, 2.0
vdw_point_density : 1.0
esp               : 1_bromobutane_conf1_a_esp.dat,
                    1_bromobutane_conf1_b_esp.dat,
                    1_bromobutane_conf1_c_esp.dat,
                    1_bromobutane_conf1_d_esp.dat,
                    1_bromobutane_conf2_a_esp.dat,
                    1_bromobutane_conf2_b_esp.dat,
                    1_bromobutane_conf2_c_esp.dat,
                    1_bromobutane_conf2_d_esp.dat

grid              : 1_bromobutane_conf1_a_grid.dat,
                    1_bromobutane_conf1_b_grid.dat,
                    1_bromobutane_conf1_c_grid.dat,
                    1_bromobutane_conf1_d_grid.dat,
                    1_bromobutane_conf2_a_grid.dat,
                    1_bromobutane_conf2_b_grid.dat,
                    1_bromobutane_conf2_c_grid.dat,
                    1_bromobutane_conf2_d_grid.dat

vdw_radii_set     : legacy
vdw_radii         : BR = 1.85

[hyperbolic.restraint.options]
weight            : 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0
restraint         : True
resp_a            : 0.001
resp_b            : 0.1
ihfree            : True
toler             : 1e-5
max_it            : 50 

[constraints]
constraint_charge : 14 = -0.20276883

equivalent_groups : group_1 = 2 3 4,
                    group_2 = 6 7,
                    group_3 = 9 10,
                    group_4 = 12 13

[qm.options]
method_esp    : None
basis_esp     : None

formal_charge : 0
multiplicity  : 1
'''

with open('stage_2.ini', 'w') as file:
    file.write(stage_2)

In [12]:
stage_2_x = f'''\
[molecules]
input_files: 1_bromobutane_conf1_x_a.xyz,
             1_bromobutane_conf1_x_b.xyz,
             1_bromobutane_conf1_x_c.xyz,
             1_bromobutane_conf1_x_d.xyz,
             1_bromobutane_conf2_x_a.xyz,
             1_bromobutane_conf2_x_b.xyz,
             1_bromobutane_conf2_x_c.xyz,
             1_bromobutane_conf2_x_d.xyz
    
[vdw.surface.options]
vdw_scale_factors : 1.4, 1.6, 1.8, 2.0
vdw_point_density : 1.0
esp               : 1_bromobutane_conf1_a_esp.dat,
                    1_bromobutane_conf1_b_esp.dat,
                    1_bromobutane_conf1_c_esp.dat,
                    1_bromobutane_conf1_d_esp.dat,
                    1_bromobutane_conf2_a_esp.dat,
                    1_bromobutane_conf2_b_esp.dat,
                    1_bromobutane_conf2_c_esp.dat,
                    1_bromobutane_conf2_d_esp.dat

grid              : 1_bromobutane_conf1_a_grid.dat,
                    1_bromobutane_conf1_b_grid.dat,
                    1_bromobutane_conf1_c_grid.dat,
                    1_bromobutane_conf1_d_grid.dat,
                    1_bromobutane_conf2_a_grid.dat,
                    1_bromobutane_conf2_b_grid.dat,
                    1_bromobutane_conf2_c_grid.dat,
                    1_bromobutane_conf2_d_grid.dat

vdw_radii_set     : legacy
vdw_radii         : BR = 1.85,
                    X = 0.0

[hyperbolic.restraint.options]
weight            : 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0
restraint         : True
resp_a            : 0.0005
resp_b            : 0.1
ihfree            : True
toler             : 1e-5
max_it            : 50 

[constraints]
constraint_charge : 14 = -0.41354762,
                    15 =  0.10314887
                    
equivalent_groups : group_1 = 2 3 4,
                    group_2 = 6 7,
                    group_3 = 9 10,
                    group_4 = 12 13

[qm.options]
method_esp : None
basis_esp  : None

formal_charge : 0
multiplicity  : 1
'''

with open('stage_2_x.ini', 'w') as file:
    file.write(stage_2_x)

In [13]:
charges_stage_2 = driver.resp(input_ini='stage_2.ini')
print()
print(f'ESP:\n{charges_stage_2[0]}\n')
print(f'RESP Stage 2:\n{charges_stage_2[1]}')


Determining Partial Atomic Charges

INPUT COORDS, ANGSTROMS
 [[-3.91966085  0.13035288 -0.25795093]
 [-4.74768397  0.48149055 -0.86502369]
 [-4.01597049 -0.94643743 -0.15602488]
 [-4.02396557  0.56750725  0.73056208]
 [-2.57911804  0.50662453 -0.88838478]
 [-2.52675434  1.58639862 -1.00772124]
 [-2.51881864  0.08285356 -1.88822254]
 [-1.38654736  0.02487902 -0.05501505]
 [-1.42860031 -1.05333239  0.06020295]
 [-1.43653557  0.45157596  0.94148564]
 [-0.0679235   0.41186001 -0.70396561]
 [ 0.0550455  -0.03724304 -1.67646333]
 [ 0.04702298  1.48067197 -0.78759998]
 [ 1.47236279 -0.19664039  0.34908209]]
Points: 827; Coord: 14
INPUT COORDS, ANGSTROMS
 [[-1.45656773  0.13035288  3.64810777]
 [-2.28980145  0.48149055  4.24800876]
 [-1.3893916  -0.94643743  3.77120056]
 [-0.54866792  0.56750725  4.05277477]
 [-1.64189543  0.50662453  2.17836102]
 [-1.73920989  1.58639862  2.09168319]
 [-2.57416412  0.08285356  1.81204603]
 [-0.48078912  0.02487902  1.30168432]
 [-0.38420537 -1.05333239  1.37

In [14]:
charges_stage_2_x = driver.resp(input_ini='stage_2_x.ini')
print()
print(f'ESP:\n{charges_stage_2_x[0]}\n')
print(f'RESP Stage 2:\n{charges_stage_2_x[1]}')


Determining Partial Atomic Charges

INPUT COORDS, ANGSTROMS
 [[-3.91966085  0.13035288 -0.25795093]
 [-4.74768397  0.48149055 -0.86502369]
 [-4.01597049 -0.94643743 -0.15602488]
 [-4.02396557  0.56750725  0.73056208]
 [-2.57911804  0.50662453 -0.88838478]
 [-2.52675434  1.58639862 -1.00772124]
 [-2.51881864  0.08285356 -1.88822254]
 [-1.38654736  0.02487902 -0.05501505]
 [-1.42860031 -1.05333239  0.06020295]
 [-1.43653557  0.45157596  0.94148564]
 [-0.0679235   0.41186001 -0.70396561]
 [ 0.0550455  -0.03724304 -1.67646333]
 [ 0.04702298  1.48067197 -0.78759998]
 [ 1.47236279 -0.19664039  0.34908209]
 [ 2.49264568 -0.59970997  1.04661901]]
Points: 827; Coord: 15
INPUT COORDS, ANGSTROMS
 [[-1.45656773  0.13035288  3.64810777]
 [-2.28980145  0.48149055  4.24800876]
 [-1.3893916  -0.94643743  3.77120056]
 [-0.54866792  0.56750725  4.05277477]
 [-1.64189543  0.50662453  2.17836102]
 [-1.73920989  1.58639862  2.09168319]
 [-2.57416412  0.08285356  1.81204603]
 [-0.48078912  0.02487902  1.30